# 🚀 Fine-Tuning Qwen2.5-0.5B for Sub-100ms Multilingual RAG (English, Hindi, Tamil)

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** using QLoRA on a multilingual RAG dataset containing Hindi, Tamil, and English grounded context-answer triplets.

### Estimated Time on Free Colab T4 GPU: **~8 to 12 minutes**

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes

In [ ]:
# Step 2: Upload or Download the SFT Dataset
# If you have rag_sft_dataset.jsonl locally, upload it to the Files panel on the left.
import os
if not os.path.exists('rag_sft_dataset.jsonl'):
    print('Please upload rag_sft_dataset.jsonl in the Colab file browser.')
else:
    print('Dataset found:', os.path.getsize('rag_sft_dataset.jsonl') / 1024 / 1024, 'MB')

In [ ]:
# Step 3: Run Fine-Tuning Script
import json, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DATASET_PATH = 'rag_sft_dataset.jsonl'
OUTPUT_DIR = 'qwen2.5_0.5b_indic_rag_lora'
MERGED_DIR = 'qwen2.5_0.5b_indic_rag_merged'

# Load dataset
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
dataset = dataset.shuffle(seed=42).train_test_split(test_size=0.05)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}

formatted_train = dataset['train'].map(format_chat_template)
formatted_eval = dataset['test'].map(format_chat_template)

# Load 4-bit model
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

# LoRA Config
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training Args
training_args = TrainingArguments(output_dir=OUTPUT_DIR, per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type='cosine', num_train_epochs=3, logging_steps=25, eval_strategy='epoch', save_strategy='epoch', fp16=True, optim='paged_adamw_8bit', report_to='none', save_total_limit=1)

trainer = SFTTrainer(model=model, train_dataset=formatted_train, eval_dataset=formatted_eval, dataset_text_field='text', max_seq_length=512, tokenizer=tokenizer, args=training_args)
trainer.train()

In [ ]:
# Step 4: Merge LoRA Weights & Save Standalone Model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
from peft import PeftModel
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR).merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Zip the merged model for easy download
!zip -r qwen2.5_0.5b_indic_rag_merged.zip qwen2.5_0.5b_indic_rag_merged
print('✅ Saved and zipped merged model as qwen2.5_0.5b_indic_rag_merged.zip!')

In [ ]:
# Step 5 (Optional): Push directly to your Hugging Face Account
# from huggingface_hub import login
# login() # Paste your HF token
# merged_model.push_to_hub('your-username/qwen2.5-0.5b-indic-rag')
# tokenizer.push_to_hub('your-username/qwen2.5-0.5b-indic-rag')